# Etna Dataset Construction

This notebook builds the Etna case-study dataset for the Cause–Trigger analysis. Waveform features are extracted on an hourly grid, then merged with gas and meteorological context variables. The final dataset is used later for HMML/PCMCI-based trigger analysis.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from obspy.clients.fdsn import Client

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src/etna"
sys.path.append(str(SRC_DIR))

from etna_config import (
    ETNA_WAVEFORM_CONFIG,
    ETNA_GAS_METEO_COLS,
    ETNA_EVENT_TIME,
)

from etna_waveform import build_station_waveform_dataset

from etna_dataset import (
    load_etnagas_csv,
    extract_plume_co2so2_xls,
    create_etna_final_dataset,
)

from etna_plotting_utils import (
    dataset_health_report,
    plot_scaled_dataset,
    plot_variable_pdfs,
    distribution_summary,
    run_teleseismic_checks,
    plot_etna_data_with_event,
)

client = Client("INGV")
cfg = ETNA_WAVEFORM_CONFIG

## 1. Station and channel metadata

We first inspect the available INGV metadata for the Etna stations. The vertical `EHZ` channel is used for waveform feature extraction because the trigger/response features are based on vertical ground motion.

In [ ]:
# identify the station metadata for ME01/ME02, get the exact vertical channel
inventory = client.get_stations(station="ME01" # or "ME02"
                                ,level="response",
                                starttime = "2008-04-12 00:00:00.000",endtime = "2008-05-13 00:00:00.000")
print(inventory)

In [ ]:
network = inventory[0]
station = network[0] 
num_channels = len(station)
print ('number of channels: ', num_channels)
print(station)

In [ ]:
channel = [c for c in station if c.code == "EHZ"][0]
print(channel)

## 2. Hourly waveform feature extraction

For each station, waveform data are processed in daily chunks with padding for filter stability. Three log-transformed RMS features are extracted:

- `T_log`: low-frequency teleseismic trigger band, aggregated by hourly maximum.
- `S_log`: background/state band, aggregated by hourly mean.
- `Y_log`: high-frequency response band, aggregated by hourly maximum.

In [ ]:
REDOWNLOAD = False # set to True to force redownloading and reprocessing of the waveform data, which is time consuming. Set to False to load from the cached pickle files.

me01_wave, me01_failures = build_station_waveform_dataset(
    client=client,
    station="ME01",
    cfg=cfg,
    cache_path="../etna_data/etna_me01_waveform_features.pkl",
    redownload=REDOWNLOAD,
)

me02_wave, me02_failures = build_station_waveform_dataset(
    client=client,
    station="ME02",
    cfg=cfg,
    cache_path="../etna_data/etna_me02_waveform_features.pkl",
    redownload=REDOWNLOAD,
)

## 4. Gas and meteorological context variables

We add slower contextual variables from ETNAGAS and plume observations. Rain is excluded from the first-pass dataset because it is sparse and near-constant in short event windows. These variables are merged onto the hourly waveform grid.

- WindSpeed 
- Patm_3 
- AirTemp_3 
- CO2_3 
- plume data (SO2/CO2 ratio)

In [ ]:
plume_df = extract_plume_co2so2_xls("../data_etna/1012-1_VolcanicGas_Etna.xls")

In [ ]:
etnagas_df = load_etnagas_csv(
    path="../data_etna/3c.csv",
    value_cols=ETNA_GAS_METEO_COLS,
)

display(etnagas_df.head())
display(etnagas_df.isna().mean().sort_values())

## 5. Merge, scale, and save final hourly datasets

The waveform, gas, meteorological, and plume variables are merged by timestamp. Raw variables are retained, and scaled versions are produced for causal discovery. The output is saved separately for ME01 and ME02.

In [ ]:
final_me01_raw, final_me01_scaled = create_etna_final_dataset(
    wave_df=me01_wave,
    station_name="ME01",
    out_csv="../data_etna/FINAL_ME01.csv",
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_GAS_METEO_COLS,
    plume_df=plume_df,
)

final_me02_raw, final_me02_scaled = create_etna_final_dataset(
    wave_df=me02_wave,
    station_name="ME02",
    out_csv="../data_etna/FINAL_ME02.csv",
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_GAS_METEO_COLS,
    plume_df=plume_df,
)

In [ ]:
# load saved datasets for analysis
#final_me01_raw = pd.read_csv("../data_etna/FINAL_ME01_raw.csv", parse_dates=["timestamp"]).set_index("timestamp")
#final_me01_scaled = pd.read_csv("../data_etna/FINAL_ME01_scaled.csv", parse_dates=["timestamp"]).set_index("timestamp")
#final_me02_raw = pd.read_csv("../data_etna/FINAL_ME02_raw.csv", parse_dates=["timestamp"]).set_index("timestamp")
#final_me02_scaled = pd.read_csv("../data_etna/FINAL_ME02_scaled.csv", parse_dates=["timestamp"]).set_index("timestamp")

## 6. Dataset quality checks

We check dataset size, timestamp order, duplicate timestamps, missing values, and variable distributions. These checks ensure that the final hourly dataset is suitable for causal discovery.

In [ ]:
station_data = {
    "ME01": {
        "wave": me01_wave,
        "final_raw": final_me01_raw,
        "final_scaled": final_me01_scaled,
    },
    "ME02": {
        "wave": me02_wave,
        "final_raw": final_me02_raw,
        "final_scaled": final_me02_scaled,
    },
}

In [ ]:
for sta, d in station_data.items():
    dataset_health_report(d["final_raw"], f"{sta} hourly raw")
    dataset_health_report(d["final_scaled"], f"{sta} hourly scaled")

## 7. Time-series inspection

The scaled variables are plotted to inspect temporal structure, outliers, and station-level differences. This helps identify whether the extracted variables behave consistently across ME01 and ME02.

In [ ]:
for sta, d in station_data.items():
    plot_scaled_dataset(d["final_scaled"], sta)

## 8. Distribution diagnostics

Empirical density plots and summary statistics are used to inspect skewness, outliers, and scaling behavior. 

In [ ]:
plot_variable_pdfs(final_me01_raw, "ME01 hourly")
plot_variable_pdfs(final_me02_raw, "ME02 hourly")

#### distribution summary table

In [ ]:
summary_me01 = distribution_summary(final_me01_raw, "ME01 hourly")
summary_me02 = distribution_summary(final_me02_raw, "ME02 hourly")

display(summary_me01)
display(summary_me02)

## 9. Teleseismic arrival diagnostics

We inspect the waveform around the Wenchuan earthquake arrival. The 1-minute RMS is compared with hourly maximum aggregation to verify that the hourly grid preserves the arrival-period energy. The spectrogram confirms that the selected frequency bands capture the relevant trigger and response components.

In [ ]:
results = {}

for station in ["ME01", "ME02"]:
    results[station] = run_teleseismic_checks(
        client=client,
        station=station,
        cfg=ETNA_WAVEFORM_CONFIG,
        event_time=ETNA_EVENT_TIME,
    )

## 10. Final hourly dataset with event marker

The final dataset is plotted with the teleseismic event time marked. This provides a visual check that the causal-analysis dataset aligns with the known event window.

In [ ]:
plot_etna_data_with_event(
    "../data_etna/FINAL_ME01_raw.csv",
    station="ME01",
    event_time=ETNA_EVENT_TIME,
    title="Etna monitoring variables, station ME01",
)